# Session 15 — End-to-End MLOps Pipeline for Smart Healthcare Monitoring

**Goal:** a capstone tying together everything so far — train, track (MLflow), check
data integrity (Deepchecks), serve (FastAPI), and monitor for drift (Evidently) — on
a single real dataset, fully runnable locally with no cloud account.

This uses the same heart disease dataset as `5. MLOps/2. End-to-End ML/heart_disease.ipynb`
and `8. CodeAlongs/4. MLFlow/mlflow_experiment_tracking.ipynb`, but wires the full
pipeline together instead of demonstrating one tool at a time.

## Prerequisites

```bash
pip install mlflow deepchecks evidently fastapi
```
Runs entirely locally.

## Step 1 — Load and integrity-check the data before training

Following Session 11's pattern: never train on data you haven't validated.

In [ ]:
import pandas as pd
import numpy as np
from deepchecks.tabular import Dataset
from deepchecks.tabular.suites import data_integrity

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(df.shape)

ds = Dataset(df, label="target", cat_features=[])
integrity_result = data_integrity().run(ds)
all_passed = integrity_result.passed(fail_if_warning=False)
print(f"Data integrity checks passed: {all_passed}")

if not all_passed:
    raise SystemExit("Data integrity check failed -- fix the data before training.")

## Step 2 — Train and track with MLflow

Same tracked-training pattern as Session 1, on real clinical data this time.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session15-healthcare-monitoring")

X, y = df.drop(columns="target"), df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run(run_name="healthcare_rf") as run:
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("test_auc", auc)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.sklearn.log_model(model, artifact_path="model")

    run_id = run.info.run_id
    print(f"run_id={run_id}  test_auc={auc:.4f}  test_accuracy={acc:.4f}")

## Step 3 — Serve the tracked model behind a FastAPI endpoint

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

app = FastAPI(title="Healthcare Risk Monitor")

class PatientFeatures(BaseModel):
    age: float; sex: float; cp: float; trestbps: float; chol: float
    fbs: float; restecg: float; thalach: float; exang: float
    oldpeak: float; slope: float; ca: float; thal: float

@app.post("/v1/predict-risk")
def predict_risk(features: PatientFeatures):
    row = np.array([[getattr(features, c) for c in X.columns]])
    risk_proba = loaded_model.predict_proba(row)[0, 1]
    return {"heart_disease_risk_probability": round(float(risk_proba), 4)}

client = TestClient(app)
sample = X_test.iloc[0].to_dict()
response = client.post("/v1/predict-risk", json=sample)
print(response.status_code, response.json())

## Step 4 — Simulate incoming patient monitoring data, check for drift

In production, a hospital feeds a steady stream of new patient records through this
endpoint. Session 5's drift-detection pattern applies directly: compare a recent
window of "current" patients against the training reference.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

rng = np.random.default_rng(7)
reference = X_train.copy()
reference["target"] = y_train.values

current = X_test.copy()
current["target"] = y_test.values
# Simulate an aging patient population this month (a realistic, gradual drift)
current["age"] = current["age"] + rng.integers(3, 8, size=len(current))

drift_report = Report([DataDriftPreset()])
snapshot = drift_report.run(current_data=current, reference_data=reference)
snapshot_result = snapshot.dict()

drift_count_metric = snapshot_result["metrics"][0]
n_drifted = int(drift_count_metric["value"]["count"])
drift_share = drift_count_metric["value"]["share"]
n_columns = len(current.columns)
drift_result = {"dataset_drift": drift_share >= 0.5,
                 "number_of_drifted_columns": n_drifted,
                 "number_of_columns": n_columns}

print(f"Dataset drift detected: {drift_result['dataset_drift']}")
print(f"Drifted columns: {drift_result['number_of_drifted_columns']}/{drift_result['number_of_columns']}")

## Step 5 — Alert logic: what happens when drift is detected

A minimal version of the alerting a real monitoring system would run — in production
this would page an on-call engineer or open a ticket instead of printing.

In [ ]:
def check_and_alert(drift_result, auc_threshold_run_id, X_test, y_test, model):
    current_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    alerts = []

    if drift_result["dataset_drift"]:
        alerts.append(f"DATA DRIFT: {drift_result['number_of_drifted_columns']} columns drifted")
    if current_auc < 0.80:
        alerts.append(f"PERFORMANCE: test AUC dropped to {current_auc:.4f} (threshold 0.80)")

    if alerts:
        print("ALERTS:")
        for a in alerts:
            print(f"  - {a}")
        print("  -> Session 17 shows how to turn this into an automatic retraining trigger.")
    else:
        print("No alerts. Model and data look healthy.")

check_and_alert(drift_result, run_id, X_test, y_test, model)

## What to try next

* Wrap Steps 1-2 as the Deepchecks-then-train pattern in a scheduled job, and Steps
  4-5 as a separate nightly monitoring job, following Session 12's pipeline-graph
  approach.
* Session 17 completes the loop: instead of just alerting, automatically retrain and
  redeploy when drift crosses a threshold.
* Add a fairness check across a sensitive attribute (e.g. `sex`) using Deepchecks'
  weak-segment detection from Session 11, since healthcare models carry real
  consequences for underperforming subgroups.